# Data Lake - Olimpíadas\n\nPipeline para construção das camadas raw e bronze do Data Lake olímpico.

In [1]:
import pandas as pd
import json
import shutil
import os
from pathlib import Path
from datetime import datetime

In [2]:
BASE_DIR = Path("olympics-datalake")
RAW_DIR = BASE_DIR / "raw"
BRONZE_DIR = BASE_DIR / "bronze"
GOLD_DIR = BASE_DIR / "gold"

SOURCE_HISTORICO = "dados-historicos/world_olympedia_olympics_athlete_event_result.csv"
SOURCE_PARIS = "dados-paris-2024/medallists.csv"

for d in [RAW_DIR, BRONZE_DIR,
          GOLD_DIR / "analise_medalhas",
          GOLD_DIR / "analise_modalidades",
          GOLD_DIR / "analise_genero"]:
    d.mkdir(parents=True, exist_ok=True)


def criar_metadados(nome, fonte, descricao, colunas, qtd_linhas, formato, observacoes=""):
    return {
        "nome": nome,
        "fonte": fonte,
        "descricao": descricao,
        "formato": formato,
        "colunas": colunas,
        "quantidade_linhas": qtd_linhas,
        "data_coleta": datetime.now().strftime("%Y-%m-%d"),
        "observacoes": observacoes
    }


def salvar_metadados(metadados, caminho):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=4)

## Camada Raw\n\nDados brutos copiados para o Data Lake com metadados JSON.

In [3]:
shutil.copy2(SOURCE_HISTORICO, RAW_DIR / "olympics_historico.csv")
df_hist = pd.read_csv(RAW_DIR / "olympics_historico.csv", low_memory=False)

meta_hist = criar_metadados(
    nome="Olympics Histórico",
    fonte="Kaggle - World Olympedia",
    descricao="Dados de atletas e resultados de todas as edições dos Jogos Olímpicos",
    colunas=[{"nome": col, "tipo": str(df_hist[col].dtype)} for col in df_hist.columns],
    qtd_linhas=len(df_hist),
    formato="csv",
    observacoes="Contém dados desde 1896 até 2022, incluindo esportes de verão e inverno"
)
salvar_metadados(meta_hist, RAW_DIR / "olympics_historico.json")

print(f"Histórico: {len(df_hist)} linhas, {len(df_hist.columns)} colunas")
df_hist.head()

Histórico: 316834 linhas, 11 colunas


,edition,edition_id,country_noc,sport,event,result_id,athlete,athlete_id,position,medal,is_team_sport
0,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Willy von Eschen,98710,Did not finish,NaN,False
1,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,"Jean, Comte de Beaumont",42118,Did not start,NaN,False
2,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,Pierre Dormeuil,85267,Did not finish,NaN,False
3,1928 Winter Olympics,30,GBR,Skeleton,"Skeleton, Men",1,Lord Brabazon of Tara,1202561,Did not start,NaN,False
4,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Alexander Berner,84063,5,NaN,False


In [4]:
shutil.copy2(SOURCE_PARIS, RAW_DIR / "olympics_paris2024.csv")
df_paris = pd.read_csv(RAW_DIR / "olympics_paris2024.csv")

meta_paris = criar_metadados(
    nome="Olympics Paris 2024",
    fonte="Kaggle - Paris 2024 Olympic Games",
    descricao="Dados dos medalhistas dos Jogos Olímpicos de Paris 2024",
    colunas=[{"nome": col, "tipo": str(df_paris[col].dtype)} for col in df_paris.columns],
    qtd_linhas=len(df_paris),
    formato="csv",
    observacoes="Dados oficiais das Olimpíadas de Paris 2024"
)
salvar_metadados(meta_paris, RAW_DIR / "olympics_paris2024.json")

print(f"Paris 2024: {len(df_paris)} linhas, {len(df_paris.columns)} colunas")
df_paris.head()

Paris 2024: 2315 linhas, 21 colunas


,medal_date,medal_type,medal_code,name,gender,country_code,country,country_long,nationality_code,nationality,...,team,team_gender,discipline,event,event_type,url_event,birth_date,code_athlete,code_team,is_medallist
0,2024-07-27,Gold Medal,1.0,EVENEPOEL Remco,Male,BEL,Belgium,Belgium,BEL,Belgium,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,2000-01-25,1903136,NaN,True
1,2024-07-27,Silver Medal,2.0,GANNA Filippo,Male,ITA,Italy,Italy,ITA,Italy,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1996-07-25,1923520,NaN,True
2,2024-07-27,Bronze Medal,3.0,van AERT Wout,Male,BEL,Belgium,Belgium,BEL,Belgium,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1994-09-15,1903147,NaN,True
3,2024-07-27,Gold Medal,1.0,BROWN Grace,Female,AUS,Australia,Australia,AUS,Australia,...,NaN,NaN,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1992-07-07,1940173,NaN,True
4,2024-07-27,Silver Medal,2.0,HENDERSON Anna,Female,GBR,Great Britain,Great Britain,GBR,Great Britain,...,NaN,NaN,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1998-11-14,1912525,NaN,True


## Camada Bronze\n\nConversão para Parquet e integração entre as fontes de dados.

Conversão dos CSVs originais para Parquet.

In [5]:
df_hist.to_parquet(BRONZE_DIR / "olympics_historico.parquet", engine="pyarrow", index=False)

meta_hist_pq = criar_metadados(
    nome="Olympics Histórico",
    fonte="Camada raw - olympics_historico.csv",
    descricao="Dados históricos convertidos para formato Parquet",
    colunas=[{"nome": col, "tipo": str(df_hist[col].dtype)} for col in df_hist.columns],
    qtd_linhas=len(df_hist),
    formato="parquet"
)
salvar_metadados(meta_hist_pq, BRONZE_DIR / "olympics_historico.json")

df_paris.to_parquet(BRONZE_DIR / "olympics_paris2024.parquet", engine="pyarrow", index=False)

meta_paris_pq = criar_metadados(
    nome="Olympics Paris 2024",
    fonte="Camada raw - olympics_paris2024.csv",
    descricao="Dados de Paris 2024 convertidos para formato Parquet",
    colunas=[{"nome": col, "tipo": str(df_paris[col].dtype)} for col in df_paris.columns],
    qtd_linhas=len(df_paris),
    formato="parquet"
)
salvar_metadados(meta_paris_pq, BRONZE_DIR / "olympics_paris2024.json")

print("Arquivos Parquet criados")

Arquivos Parquet criados


Integração entre histórico e Paris 2024 - dataset unificado de medalhas.

In [6]:
medalhas_hist = df_hist[df_hist["medal"].isin(["Gold", "Silver", "Bronze"])].copy()
medalhas_hist["ano"] = medalhas_hist["edition"].str[:4].astype(int)
medalhas_hist = medalhas_hist.rename(columns={"country_noc": "pais", "athlete": "atleta", "medal": "medalha"})
medalhas_hist = medalhas_hist[["ano", "edition", "pais", "sport", "event", "atleta", "medalha"]]

medalhas_paris = df_paris.copy()
medalhas_paris["medalha"] = medalhas_paris["medal_type"].str.replace(" Medal", "")
medalhas_paris["ano"] = 2024
medalhas_paris["edition"] = "2024 Summer Olympics"
medalhas_paris = medalhas_paris.rename(columns={"country_code": "pais", "name": "atleta", "discipline": "sport"})
medalhas_paris = medalhas_paris[["ano", "edition", "pais", "sport", "event", "atleta", "medalha"]]

df_medalhas = pd.concat([medalhas_hist, medalhas_paris], ignore_index=True)

df_medalhas.to_parquet(BRONZE_DIR / "medalhas_1986_2024.parquet", engine="pyarrow", index=False)
df_medalhas.to_csv(BRONZE_DIR / "medalhas_1986_2024.csv", index=False)

meta_medalhas = criar_metadados(
    nome="Medalhas 1986-2024",
    fonte="Integração olympics_historico + olympics_paris2024",
    descricao="Dataset unificado de todos os medalhistas olímpicos",
    colunas=[{"nome": col, "tipo": str(df_medalhas[col].dtype)} for col in df_medalhas.columns],
    qtd_linhas=len(df_medalhas),
    formato="parquet/csv",
    observacoes="Join entre dados históricos e Paris 2024, apenas medalhistas"
)
salvar_metadados(meta_medalhas, BRONZE_DIR / "medalhas_1986_2024.json")

print(f"Medalhas: {len(df_medalhas)} registros")
df_medalhas.head()

Medalhas: 47002 registros


,ano,edition,pais,sport,event,atleta,medalha
0,1928,1928 Winter Olympics,USA,Skeleton,"Skeleton, Men",Jennison Heaton,Gold
1,1948,1948 Winter Olympics,ITA,Skeleton,"Skeleton, Men",Nino Bibbia,Gold
2,2002,2002 Winter Olympics,USA,Skeleton,"Skeleton, Men","Jim Shea, Jr.",Gold
3,2002,2002 Winter Olympics,USA,Skeleton,"Skeleton, Women",Tristan Gale,Gold
4,2006,2006 Winter Olympics,CAN,Skeleton,"Skeleton, Men",Duff Gibson,Gold


Contagem de medalhas por modalidade e edição.

In [7]:
df_modalidades = df_medalhas.pivot_table(
    index=["ano", "edition", "sport"],
    columns="medalha",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()

for col in ["Gold", "Silver", "Bronze"]:
    if col not in df_modalidades.columns:
        df_modalidades[col] = 0

df_modalidades = df_modalidades[["ano", "edition", "sport", "Gold", "Silver", "Bronze"]]
df_modalidades["Total"] = df_modalidades[["Gold", "Silver", "Bronze"]].sum(axis=1)
df_modalidades.columns.name = None

df_modalidades.to_parquet(BRONZE_DIR / "modalidades_1986_2024.parquet", engine="pyarrow", index=False)
df_modalidades.to_csv(BRONZE_DIR / "modalidades_1986_2024.csv", index=False)

meta_mod = criar_metadados(
    nome="Modalidades 1986-2024",
    fonte="Integração medalhas_1986_2024",
    descricao="Contagem de medalhas por esporte e edição olímpica",
    colunas=[{"nome": col, "tipo": str(df_modalidades[col].dtype)} for col in df_modalidades.columns],
    qtd_linhas=len(df_modalidades),
    formato="parquet/csv"
)
salvar_metadados(meta_mod, BRONZE_DIR / "modalidades_1986_2024.json")

print(f"Modalidades: {len(df_modalidades)} registros")
df_modalidades.head()

Modalidades: 1116 registros


,ano,edition,sport,Gold,Silver,Bronze,Total
0,1896,1896 Summer Olympics,Artistic Gymnastics,26,9,6,41
1,1896,1896 Summer Olympics,Athletics,12,13,12,37
2,1896,1896 Summer Olympics,Cycling Road,1,1,1,3
3,1896,1896 Summer Olympics,Cycling Track,5,5,3,13
4,1896,1896 Summer Olympics,Fencing,3,3,3,9


Medalhistas classificados por gênero.

In [8]:
def extrair_genero(evento):
    evento = str(evento).lower()
    if "women" in evento or "female" in evento:
        return "Female"
    elif "men" in evento or "male" in evento:
        return "Male"
    return "Mixed"

atletas_hist = medalhas_hist.copy()
atletas_hist["genero"] = df_hist.loc[df_hist["medal"].isin(["Gold", "Silver", "Bronze"]), "event"].apply(extrair_genero).values
atletas_hist = atletas_hist[["ano", "edition", "pais", "sport", "atleta", "medalha", "genero"]]

atletas_paris = medalhas_paris.copy()
atletas_paris["genero"] = df_paris["gender"].values
atletas_paris = atletas_paris[["ano", "edition", "pais", "sport", "atleta", "medalha", "genero"]]

df_atletas_sexo = pd.concat([atletas_hist, atletas_paris], ignore_index=True)

df_atletas_sexo.to_parquet(BRONZE_DIR / "atletas_por_sexo.parquet", engine="pyarrow", index=False)
df_atletas_sexo.to_csv(BRONZE_DIR / "atletas_por_sexo.csv", index=False)

meta_sexo = criar_metadados(
    nome="Atletas por Sexo",
    fonte="Integração olympics_historico + olympics_paris2024",
    descricao="Medalhistas classificados por gênero",
    colunas=[{"nome": col, "tipo": str(df_atletas_sexo[col].dtype)} for col in df_atletas_sexo.columns],
    qtd_linhas=len(df_atletas_sexo),
    formato="parquet/csv",
    observacoes="Gênero extraído do nome do evento (histórico) ou coluna gender (Paris 2024)"
)
salvar_metadados(meta_sexo, BRONZE_DIR / "atletas_por_sexo.json")

print(f"Atletas por sexo: {len(df_atletas_sexo)} registros")
df_atletas_sexo["genero"].value_counts()

Atletas por sexo: 47002 registros


genero
Male      30091
Female    13992
Mixed      2919
Name: count, dtype: int64

## Metadata Schema e Resumo

In [9]:
schema = {
    "schema_version": "1.0",
    "descricao": "Schema padrão para os arquivos de metadados do Data Lake",
    "campos_obrigatorios": ["nome", "fonte", "descricao", "formato", "colunas", "quantidade_linhas", "data_coleta"],
    "tipos_campos": {
        "nome": "string - nome do dataset",
        "fonte": "string - origem dos dados",
        "descricao": "string - descrição do conteúdo",
        "formato": "string - formato do arquivo (csv, parquet, png)",
        "colunas": "array - lista de colunas com nome e tipo",
        "quantidade_linhas": "integer - número de linhas no dataset",
        "data_coleta": "string - data de criação (YYYY-MM-DD)",
        "observacoes": "string - informações adicionais (opcional)"
    }
}

salvar_metadados(schema, BASE_DIR / "metadata_schema.json")
print("metadata_schema.json criado")

metadata_schema.json criado


In [10]:
for dirpath, dirnames, filenames in sorted(os.walk(BASE_DIR)):
    nivel = dirpath.replace(str(BASE_DIR), "").count(os.sep)
    indent = "  " * nivel
    pasta = os.path.basename(dirpath)
    print(f"{indent}{pasta}/")
    for f in sorted(filenames):
        print(f"{indent}  {f}")

olympics-datalake/
  README.md
  metadata_schema.json
  bronze/
    atletas_por_sexo.csv
    atletas_por_sexo.json
    atletas_por_sexo.parquet
    medalhas_1986_2024.csv
    medalhas_1986_2024.json
    medalhas_1986_2024.parquet
    modalidades_1986_2024.csv
    modalidades_1986_2024.json
    modalidades_1986_2024.parquet
    olympics_historico.json
    olympics_historico.parquet
    olympics_paris2024.json
    olympics_paris2024.parquet
  gold/
    analise_genero/
      notebook.ipynb
    analise_medalhas/
      notebook.ipynb
    analise_modalidades/
      notebook.ipynb
  raw/
    olympics_historico.csv
    olympics_historico.json
    olympics_paris2024.csv
    olympics_paris2024.json
